# Avaliação de Utilidade

Este notebook executa a avaliação de utilidade do experimento de classificação sobre as diferentes versões do dataset gerados pelo sistema [Diferential-Privacy-Data-Pipeline](https://github.com/L-Repinaldo/Diferential-Privacy-Data-Pipeline-Experiment).

O objetivo é medir o desempenho preditivo do modelo em diferentes níveis de privacidade e gerar os artefatos utilizados posteriormente na avaliação de ataques de inferência de associação e na análise do trade-off entre utilidade e privacidade.

## 1. Objetivo do Experimento

O objetivo desta etapa é avaliar como diferentes níveis de privacidade diferencial afetam a utilidade dos dados para uma tarefa de classificação.

A utilidade é avaliada por meio do desempenho de um modelo supervisionado treinado separadamente em cada versão do dataset.

As métricas obtidas nesta etapa serão posteriormente utilizadas como referência para a análise do impacto da privatização sobre o desempenho do modelo.

## 2. Definição Experimental

Nesta seção são definidos os parâmetros que controlam a execução do experimento, incluindo o dataset utilizado, a tarefa de classificação, a variável alvo e as configurações de pré-processamento.

A definição experimental é mantida centralizada em objetos de configuração para garantir que as diferentes versões do dataset sejam avaliadas sob as mesmas condições experimentais.

### 2.1 Configuração do Experimento

A configuração abaixo define os datasets que serão utilizados, a amostra utilizada, a variável alvo e as variáveis categóricas e numéricas empregadas na preparação das features.

O alvo utilizado para a tarefa de classificação é Q006.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.core.utility_experiment_config import UtilityExperimentConfig
from src.core.dataset_config import DatasetConfig
from src.core.preprocessing_config import PreprocessingConfig
from src.core.task_config import TaskConfig


In [3]:
CATEGORICAL_COLUMNS = [
    "TP_SEXO",
    "TP_COR_RACA",
    "TP_NACIONALIDADE",
    "SG_UF_PROVA",
    "TP_ESTADO_CIVIL",
    "TP_ST_CONCLUSAO",
    "TP_ENSINO",
    "Q023",
]

ORDINAL_COLUMNS = [
    "TP_FAIXA_ETARIA",
    "Q001",
    "Q002",
    "Q003",
    "Q004",
    "Q007",
]

NUMERICAL_COLUMNS = [
    "Q005",
    "IN_TREINEIRO",
    "Q008",
    "Q009",
    "Q010",
    "Q011",
    "Q012",
    "Q013",
    "Q014",
    "Q015",
    "Q016",
    "Q017",
    "Q019",
    "Q020",
]

In [4]:
def get_classification_config():

    dataset_config = DatasetConfig(
        dataset_name="enem",
        dataset_version="enem_2025 - v-2026-09-15_00-20-22",
        data_sample_size=1_000_000,
        data_random_state=42,
    )


    preprocessing_config = PreprocessingConfig(
        categorical_columns=CATEGORICAL_COLUMNS,
        ordinal_columns=ORDINAL_COLUMNS,
        numerical_columns=NUMERICAL_COLUMNS,
    )

    return UtilityExperimentConfig(
        dataset=dataset_config,
        task=TaskConfig(task_type="classification", target="Q006"),
        preprocessing=preprocessing_config,
    )

### 2.2 Seleção do Experimento

Nesta etapa é instanciada a configuração que será utilizada na execução atual do experimento.

A partir deste ponto, as etapas seguintes utilizam essa configuração como fonte dos parâmetros experimentais.

In [5]:


evaluation_config = get_classification_config()


## 3. Preparação dos Dados

Nesta etapa são carregadas as diferentes versões do dataset que serão comparadas no experimento.

As versões são processadas utilizando as mesmas colunas, tamanho de amostra e estado aleatório definidos na configuração experimental.

O objetivo é garantir que as diferenças observadas posteriormente estejam relacionadas às diferentes versões dos dados e não a alterações na definição da tarefa.

In [6]:
from src.data.dataset_registry import load_dataset_bundle


In [7]:
dataset_bundle = load_dataset_bundle(
    dataset_name=evaluation_config.dataset.dataset_name,
    dataset_version=evaluation_config.dataset.dataset_version,
    columns=evaluation_config.preprocessing.categorical_columns 
    + evaluation_config.preprocessing.numerical_columns
    + [evaluation_config.task.target],
    sample_size=evaluation_config.dataset.data_sample_size,
    random_state=evaluation_config.dataset.data_random_state,
)

dataset_bundle

{'dataset_path': PosixPath('/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/src/data/datasets/enem/enem_2025 - v-2026-09-15_00-20-22'),
 'dataset_version': 'enem_2025 - v-2026-09-15_00-20-22',
 'datasets': [       TP_SEXO  TP_COR_RACA  TP_NACIONALIDADE SG_UF_PROVA  TP_ESTADO_CIVIL  \
  0            F            2                 2          BA                1   
  1            F            1                 1          MT                1   
  2            M            1                 1          RS                1   
  3            M            2                 1          BA                1   
  4            F            1                 1          PE                1   
  ...        ...          ...               ...         ...              ...   
  999995       F            1                 1          PB                1   
  999996       M            1                 1          MG                1   
  999997       F            1                 1    

## 4. Preparação das Features       

Nesta etapa os datasets carregados são preparados para utilização pelo modelo.

O processo inclui a separação entre features e variável alvo, divisão dos dados em conjuntos de treinamento, validação e teste e aplicação do pré-processamento definido para as variáveis categóricas e numéricas.

O mesmo procedimento é aplicado a cada versão do dataset para manter a comparabilidade entre os experimentos.

In [8]:
from src.core.splits_config import SplitConfig

split_plan = SplitConfig(seed=42, test_size=0.5)

In [9]:
from src.experiments.utility_evaluation_services import feature_preparation

prepared_features = []

for dataset_name, df in zip(dataset_bundle['dataset_names'], dataset_bundle['datasets']):
    
    prepared = feature_preparation.prepare_features(
        name=dataset_name,
        df=df,
        task_config=evaluation_config.task,
        split_plan=split_plan,
        preprocessing_config=evaluation_config.preprocessing,
    )
    prepared_features.append(prepared)


print(f"Prerpared features: {len(prepared_features)}")

Prerpared features: 7


## 5. Execução do Modelo

Esta seção concentra a configuração e a execução do modelo de classificação utilizado no experimento.

O modelo é executado separadamente para cada versão preparada do dataset, mantendo a configuração do algoritmo constante entre os experimentos.

In [10]:
from src.core.models_spec_config import ModelSpec

### 5.1 Configuração do Modelo

O modelo utilizado nesta avaliação é um classificador XGBoost configurado por meio de ModelSpec.

Os hiperparâmetros definidos abaixo permanecem constantes entre as versões do dataset para que a comparação de utilidade seja realizada sob as mesmas condições de modelagem.

In [11]:
XGBOOST_CLASSIFIER = ModelSpec(
    name="xgboost",
    model_type="xgboost_classifier",
    parameters={
        "n_estimators": 1000,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "tree_method": "hist",
        "n_jobs": 4,
        "verbosity": 0,
    },
)


In [12]:
CLASSIFICATION_MODELS = [
    XGBOOST_CLASSIFIER,
]

### 5.2 Execução

Nesta etapa o modelo é treinado e utilizado para gerar as predições necessárias à avaliação de utilidade.

Os resultados produzidos também fornecem as informações necessárias para a etapa posterior de avaliação do ataque de inferência de associação.

In [13]:
from src.experiments.utility_evaluation_services.model import model_runner

## 6. Avaliação de Utilidade

Nesta etapa são calculadas as métricas de desempenho do modelo para cada versão do dataset.

As métricas são organizadas em registros associados ao dataset, tarefa, variável alvo e modelo utilizado.

Para a classificação, são avaliadas métricas de desempenho nos conjuntos de treinamento, validação e teste, permitindo observar tanto o desempenho preditivo quanto possíveis diferenças de generalização.

Os dados auxiliares necessários para a avaliação posterior do ataque de inferência também são preparados nesta etapa.

In [14]:
from src.experiments.utility_evaluation_services import metrics

from src.core.results_config import UtilityClassificationResult

from dataclasses import asdict

utility_records = []
leakage_input= []

for prepared in prepared_features:

    models = (
        CLASSIFICATION_MODELS
        if prepared.task_type == "classification"
        else []
    )

    for model_spec in models:

        prediction = model_runner.execute_model(
            prepared_features=prepared,
            model_spec=model_spec,
        )

        utility = metrics.compute_utility_metrics(
            prediction_result=prediction,
            task_type=prepared.task_type,
        )

        record  = {
            "dataset": prepared.name,
            "task_type": prepared.task_type,
            "target": prepared.target,
            "model": model_spec.name,
            "model_type": model_spec.model_type,
        }

        record.update(asdict(utility))

        if type(utility)  == UtilityClassificationResult:

            leakage_input.append({
                "dataset": prepared.name,
                "task_type": prepared.task_type,
                "target": prepared.target,
                "model": model_spec.name,
                "model_type": model_spec.model_type,

                "X_pool": prepared.X_train,
                "y_pool": prepared.y_train,

                "target_prediction": {
                    "train_proba": prediction.train_proba,
                    "test_proba": prediction.test_proba,
                    "y_train_encoded": prediction.y_train_encoded,
                    "y_test_encoded": prediction.y_test_encoded,
                },
        })

        utility_records.append(record)


utility_records

[{'dataset': 'baseline',
  'task_type': 'classification',
  'target': 'Q006',
  'model': 'xgboost',
  'model_type': 'xgboost_classifier',
  'generalization_gap': 0.39442742482471704,
  'test_balanced_acc': 0.6819078977399163,
  'validation_balanced_acc': 0.6828696037255154,
  'train_balanced_acc': 0.6858521719881635,
  'train_precision': 0.6550460795621676,
  'validation_precision': 0.6524570844062103,
  'test_precision': 0.65177231272961,
  'train_f1': 0.6562532446028824,
  'validation_f1': 0.6533600343953971,
  'test_f1': 0.6527729587578139},
 {'dataset': 'dp_eps_0.05',
  'task_type': 'classification',
  'target': 'Q006',
  'model': 'xgboost',
  'model_type': 'xgboost_classifier',
  'generalization_gap': 0.5428137142016842,
  'test_balanced_acc': 0.6722395907511121,
  'validation_balanced_acc': 0.6728956172786082,
  'train_balanced_acc': 0.6776677278931289,
  'train_precision': 0.6478644788344532,
  'validation_precision': 0.6438094003338743,
  'test_precision': 0.643390602618188,
  

## 7. Exportação dos Resultados

Nesta etapa os resultados produzidos pelo experimento são organizados e persistidos como artefatos.

São armazenadas as métricas de utilidade, os dados auxiliares para a avaliação de leakage e os metadados necessários para identificar e reproduzir a configuração utilizada na execução.

O experiment_id é utilizado para associar os diferentes artefatos à mesma execução experimental.

In [15]:
import pandas as pd
from datetime import datetime

from artifacts.persistence import persist_utility_artifact


ARTIFACT_SCHEMA_VERSION = "1.0"

experiment_id =  datetime.now().strftime("%Y%m%d_%H%M%S")

utility_metrics = pd.DataFrame(utility_records)
utility_metrics.insert(0, "experiment_id", experiment_id)

input_leakage= pd.DataFrame(leakage_input)

if evaluation_config.task.task_type == "classification":

    EXPERIMENT_TYPE = "classification_evaluation"

    model_specs = {
        (model.name, model.model_type): {
            "name": model.name,
            "model_type": model.model_type,
            "parameters": model.parameters,
        }
        for model in [*CLASSIFICATION_MODELS]
    }


experiment_metadata = {
    "experiment_id": experiment_id,
    "experiment_type": EXPERIMENT_TYPE,
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset": {
        "name": evaluation_config.dataset.dataset_name,
        "version": evaluation_config.dataset.dataset_version,
        "sample_size": evaluation_config.dataset.data_sample_size,
        "random_state": evaluation_config.dataset.data_random_state,
    },
    "split": {
        "seed": split_plan.seed,
        "test_size": split_plan.test_size,
    },
    "preprocessing": {
        "categorical_columns": evaluation_config.preprocessing.categorical_columns,
        "numerical_columns": evaluation_config.preprocessing.numerical_columns,
    },
    "tasks": {"task_type": evaluation_config.task.task_type, "target": evaluation_config.task.target},
    "models": list(model_specs.values()),
}

artifact_path = persist_utility_artifact(
    experiment_id=experiment_id,
    metadata=experiment_metadata,
    utility_metrics=utility_metrics,
    input_leakage= input_leakage
)

artifact_path

PosixPath('/home/LR/Projects/IC Privacidade/Repositories/ML-MIA-Privacy-Evaluation/artifacts/evaluation/evaluation_20260923_164413')

## 8. Conclusão

A avaliação de utilidade produz os resultados necessários para comparar o desempenho do modelo entre as diferentes versões do dataset.

Nesta etapa são obtidas as métricas de desempenho que servirão como base para a análise posterior da relação entre utilidade e privacidade.

A interpretação do risco de vazamento não é realizada neste notebook. Os dados necessários para essa análise são persistidos e utilizados posteriormente pelo notebook de avaliação do ataque de inferência de associação.